In [1]:
from __future__ import annotations

import csv
from pathlib import Path
from typing import Iterable


In [2]:
INPUT_DIR = Path("/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections")
WHITELIST_FILE = Path("/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/first_250_npis.csv")
OUTPUT_DIR = Path("/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/connections_for_first_250_npis")
FILE_BATCH_SIZE = 100


In [3]:

source_files = sorted(INPUT_DIR.glob("*.csv"))
if not source_files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}")

print(f"Found {len(source_files)} source CSV file(s)")
source_files[:5]


Found 10 source CSV file(s)


[PosixPath('/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections/1-15-domain_merged.csv'),
 PosixPath('/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections/1-17-domain_merged.csv'),
 PosixPath('/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections/1-25-domain_merged.csv'),
 PosixPath('/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections/1-37-domain_merged.csv'),
 PosixPath('/Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/int_connections/1-40-domain_merged.csv')]

In [4]:
def batched(items: list[Path], batch_size: int) -> Iterable[list[Path]]:
    for start in range(0, len(items), batch_size):
        yield items[start : start + batch_size]


def load_whitelist(path: Path) -> list[str]:
    npis: list[str] = []

    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            if not row:
                continue

            value = row[0].strip()
            if not value or value.lower() == "npi":
                continue

            npis.append(value)

    return npis


In [5]:
whitelist = load_whitelist(WHITELIST_FILE)
if not whitelist:
    raise ValueError(f"No NPI values found in {WHITELIST_FILE}")

whitelist_set = set(whitelist)
print(f"Loaded {len(whitelist)} whitelist NPI value(s)")
whitelist[:10]


Loaded 250 whitelist NPI value(s)


['1003034927',
 '1003056300',
 '1003002783',
 '1003053653',
 '1003006271',
 '1003041484',
 '1003042342',
 '1003070541',
 '1003019514',
 '1003053422']

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

writers: dict[str, csv.writer] = {}
output_handles: dict[str, object] = {}
row_counts = {npi: 0 for npi in whitelist}
header: list[str] | None = None

try:
    for batch_number, file_batch in enumerate(batched(source_files, FILE_BATCH_SIZE), start=1):
        print(f"Processing batch {batch_number}: {len(file_batch)} file(s)")

        for csv_file in file_batch:
            print(f"  Reading {csv_file.name}")

            with csv_file.open(newline="") as input_handle:
                reader = csv.reader(input_handle)
                file_header = next(reader, None)
                if file_header is None:
                    continue

                if header is None:
                    header = file_header
                elif file_header != header:
                    raise ValueError(f"Header mismatch in {csv_file}")

                for row in reader:
                    if not row:
                        continue

                    npi = row[0].strip()
                    if npi not in whitelist_set:
                        continue

                    if npi not in writers:
                        output_path = OUTPUT_DIR / f"{npi}.csv"
                        output_handle = output_path.open("w", newline="")
                        writer = csv.writer(output_handle)
                        writer.writerow(header)
                        writers[npi] = writer
                        output_handles[npi] = output_handle

                    writers[npi].writerow(row)
                    row_counts[npi] += 1

    if header is None:
        raise ValueError("No headers found in the source CSV files")

    matched_file_count = sum(1 for count in row_counts.values() if count > 0)
    print(f"Created {matched_file_count} output file(s) in {OUTPUT_DIR}")
    print(f"{matched_file_count} NPI file(s) contain at least one matching row")
    print(f"{len(whitelist) - matched_file_count} NPI value(s) had no matching rows")
finally:
    for handle in output_handles.values():
        handle.close()

matched_npis = [npi for npi, count in row_counts.items() if count > 0]
matched_npis[:10]


Processing batch 1: 10 file(s)
  Reading 1-15-domain_merged.csv
  Reading 1-17-domain_merged.csv
  Reading 1-25-domain_merged.csv
  Reading 1-37-domain_merged.csv
  Reading 1-40-domain_merged.csv
  Reading 1-63-domain_merged.csv
  Reading 1-64-domain_merged.csv
  Reading 1-69-domain_merged.csv
  Reading 1-97-domain_merged.csv
  Reading 10-9-domain_merged.csv
Created 18 output file(s) in /Users/tomcraig/dev/mcz_notebooks_and_queries/WIN-2207/connections_for_first_250_npis
18 NPI file(s) contain at least one matching row
232 NPI value(s) had no matching rows


['1003038738',
 '1003038605',
 '1003060898',
 '1003043423',
 '1003043043',
 '1003043050',
 '1003058991',
 '1003042987',
 '1003040791',
 '1003059395']